In [1]:

import pandas as pd

data2025 = pd.read_csv('2025.csv')
data2026 = pd.read_csv('2026PartYear.csv')

print(data2025.shape)
print(data2026.shape)

(6656, 16)
(4096, 16)


In [2]:

# Combine 2025 and 2026 datasets
new = pd.concat([data2025, data2026], ignore_index=True)

print(new.shape)

(10752, 16)


In [3]:

# Filter Total U.S. and weeks 1-32

us2025 = data2025[
    (data2025['Geography'] == 'Total U.S.') &
    (data2025['Period'].between(1, 32))
].copy()

us2026 = data2026[
    (data2026['Geography'] == 'Total U.S.') &
    (data2026['Period'].between(1, 32))
].copy()

print(len(us2025))
print(len(us2026))

64
64


In [4]:

# Calculate estimated revenue
us2025['Revenue'] = us2025['Total Bulk and Bags'] * us2025['ASP Current Year']
us2026['Revenue'] = us2026['Total Bulk and Bags'] * us2026['ASP Current Year']

# Calculate volume, revenue, and weighted ASP
def summarize(df):
    volume = df['Total Bulk and Bags'].sum()
    revenue = df['Revenue'].sum()
    asp = revenue / volume

    return pd.Series({
        'Volume': volume,
        'Weighted ASP': asp,
        'Estimated Revenue': revenue
    })

results = pd.DataFrame({
    '2025': summarize(us2025),
    '2026': summarize(us2026)
})

results['YoY Change (%)'] = (
    (results['2026'] / results['2025'] - 1) * 100
)

print(results.round(2))

                           2025          2026  YoY Change (%)
Volume             1.870699e+09  2.125800e+09           13.64
Weighted ASP       1.250000e+00  1.010000e+00          -19.05
Estimated Revenue  2.329303e+09  2.142657e+09           -8.01


In [5]:

# Verify conventional and organic avocado figures

for year, df in [('2025', us2025), ('2026', us2026)]:
    print('\nYear:', year)

    for avocado_type, group in df.groupby('Type'):
        print('\n', avocado_type)
        print(summarize(group).round(2))


Year: 2025

 Conventional
Volume               1.764839e+09
Weighted ASP         1.220000e+00
Estimated Revenue    2.145036e+09
dtype: float64

 Organic
Volume               1.058595e+08
Weighted ASP         1.740000e+00
Estimated Revenue    1.842672e+08
dtype: float64

Year: 2026

 Conventional
Volume               2.020685e+09
Weighted ASP         9.800000e-01
Estimated Revenue    1.978496e+09
dtype: float64

 Organic
Volume               1.051156e+08
Weighted ASP         1.560000e+00
Estimated Revenue    1.641609e+08
dtype: float64


In [6]:

# Compare conventional and organic results by year

comparison = pd.concat({
    '2025': us2025.groupby('Type').apply(summarize),
    '2026': us2026.groupby('Type').apply(summarize)
}, names=['Year'])

print(comparison.round(3))

                         Volume  Weighted ASP  Estimated Revenue
Year Type                                                       
2025 Conventional  1.764839e+09         1.215       2.145036e+09
     Organic       1.058595e+08         1.741       1.842672e+08
2026 Conventional  2.020685e+09         0.979       1.978496e+09
     Organic       1.051156e+08         1.562       1.641609e+08


In [8]:

# Final verification of AI memo figures

print("AI MEMO VERIFICATION RESULTS")
print("----------------------------")
print("Sales Volume Change:", round(results.loc['Volume', 'YoY Change (%)'], 2), "%")
print("Weighted ASP Change:", round(results.loc['Weighted ASP', 'YoY Change (%)'], 2), "%")
print("Estimated Revenue Change:", round(results.loc['Estimated Revenue', 'YoY Change (%)'], 2), "%")

print("\nConclusion: The three national figures match the AI memo.")

AI MEMO VERIFICATION RESULTS
----------------------------
Sales Volume Change: 13.64 %
Weighted ASP Change: -19.05 %
Estimated Revenue Change: -8.01 %

Conclusion: The three national figures match the AI memo.
